<a href="https://colab.research.google.com/github/nejumi/models-initial-course/blob/main/notebooks/01_basics/02_save_code_and_outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. 迷ったら残すテンプレート

## この章のゴール
metric だけでなく、コードと出力ファイルも run に残します。

## 注目する W&B コード
```python
run.log_code(".")
run.save("outputs/*", base_path=".", policy="end")
```

自分の小さな研究 repo では `run.log_code(".")` が入口になります。この教材 repo は notebook や依存関係が多いので、実行セルでは保存範囲を `scripts/` に絞ります。

In [ ]:
# 準備セル: ローカルでは .env、Colab では Secrets から WANDB_API_KEY を読みます。
from pathlib import Path
import urllib.request

if not Path("course_utils.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/nejumi/models-initial-course/main/course_utils.py", "course_utils.py")

from course_utils import setup_wandb
PROJECT, ENTITY, WANDB_INIT = setup_wandb()

## 出力ファイルを作る
`run.save()` は、保存したいファイルを作った後に呼びます。

In [ ]:
from pathlib import Path
import json

out_dir = Path("outputs/02_save_code_and_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

(out_dir / "metrics.json").write_text(json.dumps({"train_loss": 0.31, "val_acc": 0.88}, indent=2))
(out_dir / "notes.md").write_text("# Demo output\n\nThis file was saved with `run.save`.\n")

print("Created:", sorted(p.name for p in out_dir.iterdir()))

## コードと出力ファイルを run に紐づける
このセルが、この章で覚える最小テンプレートです。

In [ ]:
import wandb

cfg = {"lr": 0.1, "epochs": 5, "model": "tiny-demo"}

with wandb.init(**WANDB_INIT, name="02-save-code-and-outputs", config=cfg, tags=["course", "save"]) as run:
    run.log({"train/loss": 0.31, "val/acc": 0.88})
    run.log_code("scripts")
    run.save("outputs/02_save_code_and_outputs/*", base_path=".", policy="end")
    run_url = run.url

print("Run URL:", run_url)

## W&B App で確認すること
- Files に `outputs/02_save_code_and_outputs/metrics.json` と `notes.md` がある
- Code に `scripts/` のコードが残っている
- API key や `.env` は notebook や git に書かない